тест ллм апи

In [ ]:
# параметры запросов
import requests

YANDEX_LLM_URL="https://ai.api.cloud.yandex.net/v1/responses"
YANDEX_LLM_MODEL="yandexgpt-5-lite"  # https://aistudio.yandex.ru/docs/ru/ai-studio/concepts/generation/models.html
YANDEX_EMBEDDING_URL="https://ai.api.cloud.yandex.net:443/foundationModels/v1/textEmbedding"

YANDEX_FOLDER_ID="Вставьте креды здесь"
YANDEX_API_KEY="Вставьте креды здесь"


headers = {
    "Authorization": f"Api-Key {YANDEX_API_KEY}",
    "Content-Type": "application/json",
}

data = {
    "model": f"gpt://{YANDEX_FOLDER_ID}/{YANDEX_LLM_MODEL}",
    "temperature": 0.8,
    "max_output_tokens": 1500
}

In [ ]:
# обращение к модели
from pprint import pprint


data["instructions"] = "Общайся как Джесси Пинкман из сериала Breaking Bad."
data["input"] = "Придумай 3 идеи для стартапа."

response = requests.post(YANDEX_LLM_URL, headers=headers, json=data)

print(response.status_code)
pprint(response.json())

200
{'background': False,
 'created_at': 1775151263.0,
 'error': None,
 'id': '38f1972e-16b1-4238-904e-f4e8c3faf5ce',
 'incomplete_details': None,
 'instructions': 'Общайся как Джесси Пинкман из сериала Breaking Bad.',
 'max_output_tokens': 1500,
 'max_tool_calls': None,
 'metadata': {},
 'model': 'gpt://b1grgbd2fpc9td1qpfu7/yandexgpt-5-lite',
 'object': 'response',
 'output': [{'content': [{'annotations': [],
                          'logprobs': None,
                          'text': 'Слушай, у меня парочка идей есть, как бы '
                                  'поднять деньжат. Например, можно замутить '
                                  'производство какого-нибудь легального '
                                  'химического добра, типа удобрений для '
                                  'фермеров. Или открыть точку по продаже '
                                  'всяких там луддитов — смесей для '
                                  'самокруток, чтобы народ расслаблялся. А ещё '
        

тест эмбединг апи

In [ ]:
# загрузка данных
import json


with open("synthetic_data.json", "r", encoding="utf-8") as f:
    data = json.load(f)

dataset_name = data["dataset_name"]
language = data["language"]
documents = data["documents"]

In [5]:
# !pip install faiss-cpu

In [ ]:
# создание эмбеддингов и индекса
import numpy as np
import faiss


def get_yandex_embedding(text):
    payload = {
        "modelUri": f"emb://{YANDEX_FOLDER_ID}/text-search-doc/latest",
        "text": text
    }
    response = requests.post(YANDEX_EMBEDDING_URL, headers=headers, json=payload)

    return response.json()["embedding"]


def enrich_text(doc):  # объединяет все поля, возможны другие методы индексации
    return f"Заголовок: {doc.get('title', '')}. Департамент: {doc.get('department', '')}. Теги: {', '.join(doc.get('tags', []))}. Содержание: {doc.get('text', '')}"

doc_texts = [enrich_text(doc) for doc in documents]

embeddings = []
for text in doc_texts:
    emb = get_yandex_embedding(text)
    if emb:
        embeddings.append(emb)

embeddings_np = np.array(embeddings).astype('float32')
dimension = embeddings_np.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(embeddings_np)

index_path = "index.faiss"
faiss.write_index(index, index_path)

In [ ]:
# ответ по документу
import faiss
import numpy as np


def search_docs(query, k=3, index_path="index.faiss"):

    index = faiss.read_index(index_path)

    query_payload = {
        "modelUri": f"emb://{YANDEX_FOLDER_ID}/text-search-query/latest",
        "text": query
    }
    response = requests.post(YANDEX_EMBEDDING_URL, headers=headers, json=query_payload)

    query_emb = response.json()["embedding"]
    query_np = np.array([query_emb]).astype('float32')

    distances, indices = index.search(query_np, k)

    print(f"Результаты для запроса: {query}\n")
    for i, idx in enumerate(indices[0]):
        doc = documents[idx]
        print(f"[{i+1}] {doc['title']} (Расстояние: {distances[0][i]:.4f})")
        print(f"{doc['text'][:150]}...")
        print("-" * 30)

search_docs("что делает юридический отдел??")

Результаты для запроса: что делает юридический отдел??

[1] Техническое задание на внутреннего юридического ассистента (Расстояние: 1.3540)
Внутренний юридический ассистент должен уметь находить по запросу сотрудников фрагменты договоров, регламентов и служебных записок, отвечать на вопрос...
------------------------------
[2] Описание API сервиса вопрос-ответ по внутренним документам (Расстояние: 1.6009)
Сервис вопрос-ответ предоставляет endpoint для постановки вопроса, выполняет поиск релевантных фрагментов по векторному индексу, собирает контекст и п...
------------------------------
[3] Инструкция по загрузке документов в систему внутреннего поиска (Расстояние: 1.6051)
При загрузке документов в систему внутреннего поиска необходимо указывать заголовок, тип документа, подразделение-владелец, дату создания, язык и набо...
------------------------------
